# Loss Functions

## 1. Regression

| Loss | PyTorch | Formula | Notes |
|---|---|---|---|
| MSE (L2) | `nn.MSELoss()` | $\frac{1}{n}\sum(\hat{y}_i - y_i)^2$ | Penalizes large errors heavily, sensitive to outliers |
| MAE (L1) | `nn.L1Loss()` | $\frac{1}{n}\sum\|\hat{y}_i - y_i\|$ | Robust to outliers, gradient is constant |
| Huber | `nn.HuberLoss(delta=1.0)` | L2 if $\|e\| \leq \delta$, L1 if $\|e\| > \delta$ | $\delta$ is a hyperparameter: best of both |
| RMSE | `torch.sqrt(nn.MSELoss()(pred, target))` | $\sqrt{\text{MSE}}$ | Same unit as $y$, easier to interpret |

> **Outliers** are the classic problem in regression. L1 and Huber are more robust than MSE.

## 2. Classification

### 2.1 `nn.CrossEntropyLoss()`

Multi-class, **one** correct class per sample. It takes the model's output, turns it into probabilities and applies $-\log$ to the probability of the correct class.

```
probs  = [0.7, 0.2, 0.1]     target = 0
loss   = -log(0.7) = 0.36
```

> PyTorch's CE already includes `softmax` + `log` internally. It expects **raw logits** and **class indices** as target. **Do not** put a softmax in the model's last layer.

### 2.2 `nn.BCEWithLogitsLoss()` / `nn.BCELoss()`

Binary classification, or **multi-label** (each output is an independent yes/no):

$$-[\,y\log(p) + (1-y)\log(1-p)\,]$$

| | `BCEWithLogitsLoss` | `BCELoss` |
|---|---|---|
| Expects | raw logits | probabilities (after sigmoid) |
| Sigmoid | already inside | you apply it in the model |
| Stability | stable | less stable |

> Prefer `BCEWithLogitsLoss`, and **do not** put a sigmoid in the model. The target must be `float`.

### 2.3 `nn.NLLLoss()`

Takes **log-probabilities** (the output of `log_softmax`), picks the one of the correct class and negates it.

```
log_probs = log([0.7, 0.2, 0.1]) = [-0.36, -1.61, -2.30]
target = 0  →  pick -0.36  →  negate  →  0.36
```

$$\text{CrossEntropyLoss} = \text{LogSoftmax} + \text{NLLLoss}$$

> The log of a number between 0 and 1 is negative, so NLL flips the sign to get a positive loss.

### Summary

| Task | Loss | Final activation in the model |
|---|---|---|
| Multi-class (one label) | `CrossEntropyLoss` | none (raw logits) |
| Binary / multi-label | `BCEWithLogitsLoss` | none (raw logits) |
| Manual setup | `NLLLoss` | `log_softmax` |

## 3. Imbalanced classes

Regression has the outlier problem, classification has **class imbalance**: the model can ignore the rare class and still get low loss. A simple fix is to **weight the loss**:

```python
# Multi-class: one weight per class
nn.CrossEntropyLoss(weight=torch.tensor([1.0, 5.0, 7.0]))
```
The loss of samples from class 0 is multiplied by 1, class 1 by 5, class 2 by 7.

```python
# Binary: weight on the positive class
nn.BCEWithLogitsLoss(pos_weight=torch.tensor([10.0]))
```
The loss of positive samples (class 1) is multiplied by 10.

> The parameter is `weight` (not `weights`). A common starting point is the inverse of the class frequency.

Other options: oversampling the rare class, undersampling the common one, or **focal loss** (down-weights easy examples).

In [24]:
import torch
import torch.nn as nn


# Regression
output = torch.randint(0, 10, (1, 10), dtype = torch.float32)
target = torch.randint(0, 10, (1, 10), dtype = torch.float32)

mse = nn.MSELoss()
mae = nn.L1Loss()
hubber = nn.HuberLoss(delta = 1)

loss_mse = mse(output, target)
loss_mae = mae(output, target)
loss_hubber = hubber(output, target)
rmse = torch.sqrt(mse(output, target))

print(f'=' *  50)
print(f'Regression Loss')
print(f'MSE: {loss_mse.item():.3f}, MAE (L1): {loss_mae.item():.3f}, Hubber: {loss_hubber.item():.3f}, RMSE: {rmse:.3f}')


# Classification
output = torch.randn(10, 100)       
target = torch.randint(0, 100, (10,))

output2 = torch.randint(0, 2, (1, 10), dtype = torch.float32)
target2 = torch.randint(0, 2, (1, 10), dtype = torch.float32)



bce = nn.BCEWithLogitsLoss()
cross = nn.CrossEntropyLoss()

loss_bce = bce(output2, target2)
loss_cross = cross(output, target)


print(f'=' *  50)
print(f'Classitication loss')
print(f'BCE: {loss_bce.item():.3f}, Cross: {loss_cross.item():.3f}')

Regression Loss
MSE: 14.500, MAE (L1): 2.900, Hubber: 2.550, RMSE: 3.808
Classitication loss
BCE: 0.865, Cross: 5.689
